In [1]:
!pip install playwright pandas beautifulsoup4 requests tqdm nest_asyncio
!playwright install

In [2]:
import requests
import pandas as pd
import nest_asyncio

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from tqdm import tqdm
from playwright.async_api import async_playwright

In [3]:
nest_asyncio.apply()

In [4]:
from urllib.parse import urljoin, urlparse

async def crawl_site_playwright(start_url, max_pages=100):

    visited = set()
    queue = [start_url]
    pages = []

    domain = urlparse(start_url).netloc

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        while queue and len(pages) < max_pages:

            url = queue.pop(0)

            if url in visited:
                continue

            visited.add(url)
            pages.append(url)

            print("Crawling:", url)

            try:

                await page.goto(url, timeout=60000)

                # grab all links from rendered page
                links = await page.eval_on_selector_all(
                    "a[href]",
                    "elements => elements.map(e => e.href)"
                )

                for link in links:

                    parsed = urlparse(link)

                    if parsed.netloc == domain:

                        clean_url = parsed.scheme + "://" + parsed.netloc + parsed.path

                        if clean_url not in visited:
                            queue.append(clean_url)

            except Exception as e:

                print("Error:", e)

        await browser.close()

    return list(set(pages))

In [5]:
async def scan_page(page, url):

    await page.goto(url)

    await page.add_script_tag(
        url="https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.8.2/axe.min.js"
    )

    results = await page.evaluate(
        """async () => {
            return await axe.run();
        }"""
    )

    return results["violations"]

In [6]:
def parse_violations(page_url, violations):

    rows = []

    for v in violations:

        for node in v["nodes"]:

            rows.append({
                "page": page_url,
                "violation": v["id"],
                "severity": v["impact"],
                "description": v["description"],
                "content": node["html"],
                "recommended_fix": v["help"]
            })

    return rows

In [7]:
pages = await crawl_site_playwright("https://www.honolulupd.org", max_pages=100)

print("Pages discovered:", len(pages))
pages

Crawling: https://www.honolulupd.org
Crawling: https://www.honolulupd.org/
Crawling: https://www.honolulupd.org/organization/
Crawling: https://www.honolulupd.org/information/
Crawling: https://www.honolulupd.org/public-affairs-office/
Crawling: https://www.honolulupd.org/community-programs/
Crawling: https://www.honolulupd.org/police-services/
Crawling: https://www.honolulupd.org/about-us/
Crawling: https://www.honolulupd.org/safer-roads-together/
Crawling: https://www.honolulupd.org/information/pedestrian-safety/
Crawling: https://www.honolulupd.org/if-you-wait-until-you-celebrate-its-too-late-plan-ahead-for-a-side-ride/
Crawling: https://www.honolulupd.org/pal/
Crawling: https://www.honolulupd.org/road-safety-during-the-storm/
Crawling: https://www.honolulupd.org/power-outage-road-safety-during-severe-weather/
Crawling: https://www.honolulupd.org/hpd-shuts-down-illegal-gambling-operation-on-mala-street-in-wahiawa-2/
Crawling: https://www.honolulupd.org/news/
Crawling: https://www.ho

['https://www.honolulupd.org/organization/patrol-districts/',
 'https://www.honolulupd.org/d6/',
 'https://www.honolulupd.org/d5/',
 'https://www.honolulupd.org/organization/patrol-districts/district-8/',
 'https://www.honolulupd.org/information/annual-report/',
 'https://www.honolulupd.org/information/traffic-information-page/',
 'https://www.honolulupd.org/information/arrest-logs/',
 'https://www.honolulupd.org/organization/chief-of-police/',
 'https://www.honolulupd.org/traffic-division/',
 'https://www.honolulupd.org/power-outage-road-safety-during-severe-weather/',
 'https://www.honolulupd.org/organization/patrol-districts/district-7/',
 'https://www.honolulupd.org/update-escapee-arrested/',
 'https://www.honolulupd.org/organization/divisions/records',
 'https://www.honolulupd.org/templates/contact/',
 'https://www.honolulupd.org/information/covered-offender-registry/',
 'https://www.honolulupd.org/information/strategic-plan/',
 'https://www.honolulupd.org/cyber-crimes/',
 'https:

In [8]:
async def run_scan(pages):

    all_rows = []
    error_rows = []

    async with async_playwright() as p:

        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for url in tqdm(pages):

            try:

                violations = await scan_page(page, url)

                rows = parse_violations(url, violations)

                all_rows.extend(rows)

            except Exception as e:

                print("Error scanning:", url)

                error_rows.append({
                    "page": url,
                    "error_message": str(e)
                })

        await browser.close()

    return all_rows, error_rows

In [9]:
all_rows, error_rows = await run_scan(pages)

 22%|█████████████████▊                                                               | 22/100 [00:18<01:09,  1.13it/s]

Error scanning: https://www.honolulupd.org/wp-content/uploads/2024/02/Pathways-Internship-Program-Brochure-4.pdf


 92%|██████████████████████████████████████████████████████████████████████████▌      | 92/100 [01:24<00:06,  1.18it/s]

Error scanning: https://www.honolulupd.org/wp-content/uploads/2026/01/Honolulu-Police-Department-2025-Legislative-Disciplinary-Report.pdf


100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [01:29<00:00,  1.12it/s]


In [14]:
violations_df = pd.DataFrame(all_rows)
errors_df = pd.DataFrame(error_rows)
pages_df = pd.DataFrame({
    "page": pages
})

In [13]:
violations_df.to_csv("wcag_violations.csv", index=False)
errors_df.to_csv("wcag_errors.csv", index=False)
pages_df.to_csv("wcag_crawled_pages.csv", index=False)

print("Reports saved!")

Reports saved!


In [12]:
#errors

#outputs

#crawle
#TEXT LEFT JUSTIFIED
#NUMBERS RIGHT